In [1]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

# ---------------------------------------------------------------------------
# Package imports (replaces ~40 inline physics functions)
# ---------------------------------------------------------------------------
from blackhole.constants import G, M_sun, c
from blackhole.disk_physics import (
    R_func,
    X_func,
    kinematic_viscosity,
)
from blackhole.evolution import (
    add_mass,
    calculate_timestep,
    disk_evap,
    evolve_surface_density,
)
from blackhole.irradiation import (
    Epsilon_irr,
    Flux_irr,
    alpha_visc_irr,
)
from blackhole.irradiation import (
    Sigma_max as dim_Sigma_max,
)
from blackhole.opacity import kappa_bath
from blackhole.solvers import (
    solve_scale_height,
    solve_temperature,
)

# ---------------------------------------------------------------------------
# Sgr A* simulation parameters (notebook-specific, NOT from package defaults)
# ---------------------------------------------------------------------------
# NOTE: This notebook uses alpha_cold=0.02 (not the package default of 0.04)
alpha_cold = 0.02
alpha_hot = 0.2

M_star = M_sun * 4.3e6   # Sgr A* mass (g)

# Disk parameters
R_1 = 4e12   # Inner radius of the disk (cm)
R_K = 1e15   # Radius where initial mass is added (cm)
R_N = 2e15   # Outer radius of the disk (cm)

# ---------------------------------------------------------------------------
# Mass transfer rate
# ---------------------------------------------------------------------------
# The DIM instability threshold at R_K = 1e15 for an SMBH is very high:
#   Sigma_max ~ 360,000 g/cm^2  (from DIM formula with alpha=0.02, M=4.3e6 M_sun)
# The cold-disk steady-state surface density is:
#   Sigma_ss = M_dot / (3*pi*nu_cold) ~ M_dot * 4.4e-17
# For Sigma_ss > Sigma_max we need M_dot > ~8e21 g/s.
# We use M_dot = 1e22 g/s (Sigma_ss/Sigma_max ~ 1.2, marginally unstable)
# which is consistent with Bondi-rate or tidal-disruption feeding models.
M_dot = 1e22  # Mass transfer rate (g/s)

min_Sigma = 1e-5

X_1 = X_func(R_1)
X_K = X_func(R_K)
X_N = X_func(R_N)

# Simulation parameters
N = 100   # Number of grid points for the simulation
N_n = 3   # Extra grid points after bath outer radius

# Define the radial grid
X = np.linspace(X_1, X_N, N)
dX = float(np.diff(X)[0])  # Uniform grid spacing
X = np.linspace(X_1, X_N + N_n * dX, N + N_n)
r = R_func(X)
dr = np.diff(r)
dr = np.insert(dr, 0, 0)
dX = float(np.diff(X)[0])  # Recalculate after extending grid
X_N = X_N + N_n * dX
N = N + N_n  # Redefine N to match new X array

# Initial conditions
Sigma = np.full(N, min_Sigma)   # Surface density array (g/cm^2)
T_c = np.full(N, 1e3)           # Mid-plane temperature array (K)
H = np.full(N, 1e7)             # Scale height array (cm)
p = np.full(N, 1e3)             # Pressure array
alpha = np.full(N, alpha_cold)   # Alpha viscosity array

nu = kinematic_viscosity(H, r, alpha, M_star)

# Mass transfer state (package add_mass returns MassTransferResult)
j_val = 0
dMj = 0.0
dMj1 = 0.0

# ---------------------------------------------------------------------------
# Auto-scale sigma_cap from Sigma_max at R_K
# ---------------------------------------------------------------------------
# For stellar-mass systems the default sigma_cap=200 is fine, but for SMBHs
# Sigma_max >> 200 and the hardcoded cap blocks mass deposition entirely.
# Use 2x Sigma_max (no irradiation, eps_irr=0) at the circularisation radius.
sigma_cap = 2.0 * dim_Sigma_max(0.0, R_K, M_star, alpha_cold)
print(f"Sigma_max at R_K = {dim_Sigma_max(0.0, R_K, M_star, alpha_cold):.0f} g/cm^2")
print(f"sigma_cap = {sigma_cap:.0f} g/cm^2")

# ---------------------------------------------------------------------------
# Timestep limits for Sgr A*
# ---------------------------------------------------------------------------
# The CFL-based timestep for SMBH scales is enormous (dt_CFL >> 1e12 s),
# so the CFL criterion never provides a useful upper bound.  Instead, we
# use dt_max_sgra as the quiescent timestep and add a thermal-timescale
# constraint that automatically reduces dt during outbursts (see sim loop).
#
# The viscous timescale is t_visc ~ R_K^2 / nu ~ 4.1e14 s (13 Myr).
# With 500k steps at dt ~ 2e9: total ~ 1e15 s ~ 2.4 t_visc (sufficient
# for multiple outburst cycles).
dt_max_sgra = 2e9    # Upper cap (quiescent timestep)
dt_min_sgra = 1e5    # Lower floor

dt = np.clip(calculate_timestep(X, nu, dX) * 10, dt_min_sgra, dt_max_sgra)

# Eddington luminosity for L_ratio scaling
L_edd = 1.4e38 * (M_star / M_sun)

# Tidal torque parameters for evolve_surface_density
# Sgr A* is an SMBH at the Galactic Centre fed by stellar winds, not a close
# binary.  a_1 must satisfy R_N / a_1 << 1 so that the tidal torque
# (r/a_1)^5 is a small correction at the outer edge.  With a_1 = 1.5e16,
# (R_N/a_1)^5 = (2e15/1.5e16)^5 = 0.13^5 = 4e-5 (negligible).
tidal_params = {"cw": 0.2, "a_1": 1.5e16, "n_1": 5, "trunc_frac": 9.3 / 10}

# Helper to compute irradiation quantities from full arrays
def compute_irr_quantities(Sigma_arr, nu_arr, r_arr):
    """Compute inner-edge irradiation quantities from full disk arrays.

    Uses the same robust threshold as find_inner_index (min_Sigma * 21)
    to prevent the inner-edge index from jumping chaotically between cells
    when Sigma oscillates near the floor.
    """
    candidates = np.where(Sigma_arr > min_Sigma * 21)[0]
    if len(candidates) == 0:
        candidates = np.where(Sigma_arr > min_Sigma)[0]
    if len(candidates) == 0:
        # Disk hasn't built up yet — zero irradiation
        zeros = np.zeros_like(r_arr)
        return zeros, zeros
    inner_idx = int(candidates[0])
    Sigma_inner = Sigma_arr[inner_idx]
    nu_inner = nu_arr[inner_idx]
    r_inner = r_arr[inner_idx]
    eps = Epsilon_irr(Sigma_inner, nu_inner, r_inner, r_arr, M_star, M_dot)
    flux = Flux_irr(Sigma_inner, nu_inner, r_inner, r_arr, M_star, M_dot)
    return eps, flux

def find_inner_index(Sigma_arr):
    """Find index of first cell with significant surface density."""
    candidates = np.where(Sigma_arr > min_Sigma * 21)[0]
    if len(candidates) == 0:
        # Disk hasn't spread yet — fall back to first cell above min_Sigma
        candidates = np.where(Sigma_arr > min_Sigma)[0]
    return int(candidates[0]) if len(candidates) > 0 else 0

def plot_s(Sigma_arr, r_arr):
    """Quick plot of a quantity vs radius."""
    plt.figure(figsize=(10, 6))
    plt.plot(np.asarray(r_arr), np.asarray(Sigma_arr))
    plt.show()

Sigma_max at R_K = 359895 g/cm^2
sigma_cap = 719789 g/cm^2


In [2]:
timesteps = 500001 # Number of timesteps to simulate
output_times = set(np.linspace(1, 500000, 100000, dtype='int64')) # Specific timesteps to output for plotting

In [ ]:
# ---------------------------------------------------------------------------
# Initialize fresh disk from cell 1 initial conditions (no CSV dependency)
# ---------------------------------------------------------------------------
totalt = 0.0

# History arrays — start with the initial state from cell 1
Sigma_history = [Sigma.copy()]
Temp_history = [T_c.copy()]
H_history = [H.copy()]
alpha_history = [alpha.copy()]
t_history = [0.0]
Sigma_transfer_history = [float(Sigma[1])]

# Save initial state
np.savetxt("../../data/Sigma_history_bath_array.csv", Sigma_history, delimiter=",")
np.savetxt("../../data/Temp_history_bath_array.csv", Temp_history, delimiter=",")
np.savetxt("../../data/H_history_bath_array.csv", H_history, delimiter=",")
np.savetxt("../../data/t_history_bath_array.csv", t_history, delimiter=",")
np.savetxt("../../data/alpha_history_bath_array.csv", alpha_history, delimiter=",")
np.savetxt("../../data/Sigma_transfer_history_bath_array.csv", Sigma_transfer_history, delimiter=",")

print(f"Fresh disk initialized: N={N}, Sigma=[{Sigma.min():.2e}, {Sigma.max():.2e}], T_c=[{T_c.min():.0f}, {T_c.max():.0f}] K")
print(f"Initial dt = {dt:.2f} s")
print("Saved to ../../data/*_history_bath_array.csv")

In [ ]:
%%time

#########################################
############### Simulation ##############
#########################################

DATA_PREFIX = "../../data/"
DATA_SUFFIX = "_history_bath_array.csv"

trunc_rad = int(N * (9.3 / 10))

for i in range(1):

    Sigma_history = pd.read_csv(DATA_PREFIX + "Sigma" + DATA_SUFFIX, header=None).to_numpy().tolist()
    Temp_history = pd.read_csv(DATA_PREFIX + "Temp" + DATA_SUFFIX, header=None).to_numpy().tolist()
    H_history = pd.read_csv(DATA_PREFIX + "H" + DATA_SUFFIX, header=None).to_numpy().tolist()
    alpha_history = pd.read_csv(DATA_PREFIX + "alpha" + DATA_SUFFIX, header=None).to_numpy().tolist()
    t_history = pd.read_csv(DATA_PREFIX + "t" + DATA_SUFFIX, header=None).to_numpy().flatten().tolist()
    Sigma_transfer_history = pd.read_csv(
        DATA_PREFIX + "Sigma_transfer" + DATA_SUFFIX, header=None
    ).to_numpy().flatten().tolist()

    totalt = t_history[-1]
    Sigma = np.array(Sigma_history[-1])
    T_c = np.array(Temp_history[-1])
    H = np.array(H_history[-1])
    nu = kinematic_viscosity(H, r, alpha, M_star)
    # Crank-Nicolson (theta=0.5) is unconditionally stable for diffusion,
    # so the CFL criterion is not a hard constraint.  For SMBH scales the
    # CFL timestep is always >> dt_max_sgra, so dt starts at the cap.
    # A thermal-timescale constraint (see below) automatically reduces dt
    # during outbursts when the heating front must be resolved.
    dt = np.clip(calculate_timestep(X, nu, dX) * 300, dt_min_sgra, dt_max_sgra)

    for timestep in range(timesteps):

        # Stage 1: Add mass at the outer radius (with auto-scaled sigma_cap)
        result = add_mass(Sigma, M_dot, dt, X, N, X_K, X_N, dX, min_Sigma,
                          sigma_cap=sigma_cap)
        Sigma = result.Sigma
        j_val = result.j_val
        dMj = result.dMj
        dMj1 = result.dMj1

        # Compute luminosity-scaled evaporation (Liu et al. 2002):
        # L_ratio = L_actual / L_edd, capped at 1.0
        M_dot_inner = 6.0 * np.pi * Sigma[1] * nu[1]
        L_actual = 0.1 * M_dot_inner * c**2
        L_ratio = min(L_actual / L_edd, 1.0) if L_edd > 0 else 0.0

        def evap_func(r_arr, _lr=L_ratio):
            return disk_evap(r_arr, M_star, L_ratio=_lr)

        # Stage 2: Evolve surface density (diffusion + conditional tidal torques + evaporation)
        # Use Crank-Nicolson (theta=0.5) for unconditional stability
        # Only apply tidal torques when j_val >= trunc_rad (preserves original notebook logic)
        tp = tidal_params if j_val >= trunc_rad else None
        Sigma = evolve_surface_density(
            Sigma, dt, nu, X, dX, N, min_Sigma,
            tidal_params=tp, evap_func=evap_func,
            theta=0.5,
        )

        # Stage 3: Compute irradiation for temperature solver
        eps_irr, F_irr_arr = compute_irr_quantities(Sigma, nu, r)

        # Update disk temperature using package solver (with irradiation flux)
        T_c = solve_temperature(H, Sigma, r, T_c, alpha, M_star,
                                F_irr=F_irr_arr, opacity_func=kappa_bath)

        # Scale height at current Sigma
        H = solve_scale_height(H, Sigma, r, T_c, M_star)

        # Alpha viscosity with irradiation-shifted critical temperature
        alpha = alpha_visc_irr(T_c, eps_irr, r, M_star,
                               alpha_cold=alpha_cold, alpha_hot=alpha_hot)

        # Update viscosity for the next timestep
        nu = kinematic_viscosity(H, r, alpha, M_star)

        # -----------------------------------------------------------------
        # Adaptive timestep with thermal-timescale constraint
        # -----------------------------------------------------------------
        # The CFL-based timestep for SMBH scales is always >> dt_max_sgra,
        # so it never limits dt.  During outbursts, the alpha(T) transition
        # must be resolved over multiple steps to prevent flip-flopping
        # between hot and cold states.  We limit dt to 20x the thermal
        # timescale (t_th = 1 / (alpha * Omega_K)) at the hottest cell.
        # In quiescence (all cells cold), this constraint is inactive and
        # dt stays at dt_max_sgra.
        dt = np.clip(calculate_timestep(X, nu, dX) * 300, dt_min_sgra, dt_max_sgra)

        alpha_max_val = float(np.max(alpha[1:-1]))
        if alpha_max_val > 2.0 * alpha_cold:
            i_hot = int(np.argmax(alpha[1:-1])) + 1
            Omega_K = np.sqrt(G * M_star / r[i_hot]**3)
            t_thermal = 1.0 / (alpha_max_val * Omega_K)
            dt = min(dt, max(20.0 * t_thermal, dt_min_sgra))

        totalt += dt

        # Store state at specific timesteps
        if timestep in output_times:
            Inner_index = find_inner_index(Sigma)
            Sigma_transfer_history.append(float(Sigma[Inner_index]))
            Sigma_history.append(Sigma.copy())
            Temp_history.append(T_c.copy())
            H_history.append(H.copy())
            alpha_history.append(alpha.copy())
            t_history.append(totalt)

    np.savetxt(DATA_PREFIX + "Sigma" + DATA_SUFFIX, Sigma_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "Temp" + DATA_SUFFIX, Temp_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "H" + DATA_SUFFIX, H_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "t" + DATA_SUFFIX, t_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "alpha" + DATA_SUFFIX, alpha_history, delimiter=",")
    np.savetxt(DATA_PREFIX + "Sigma_transfer" + DATA_SUFFIX, Sigma_transfer_history, delimiter=",")

print(f"Simulation complete: {len(t_history)} snapshots, total time = {totalt:.2e} s")